# EDA (Exploratory Data Analysis)
## 탐색적 데이터 분석
---

## 0. 라이브러리 import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 그래프 설정
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

# 출력 옵션
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('라이브러리 import 완료')

## 1. 데이터 로드

In [ ]:
# 데이터 로드
file_path = os.path.join('data', 'synthetic_customer_churn_100k.csv')
df = pd.read_csv(file_path, skipinitialspace=True)
print(f'데이터 크기: {df.shape}')
df.head()

## 2. 기본 정보 확인

In [ ]:
# 데이터 타입 및 결측치 확인
df.info()

In [ ]:
# 기술통계량
df.describe()

In [ ]:
# 범주형 컬럼 기술통계량
df.describe(include='object')

## 3. 결측치 확인

In [ ]:
# 결측치 개수
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    '결측치 개수': missing,
    '결측치 비율(%)': missing_pct
})
missing_df[missing_df['결측치 개수'] > 0]

In [ ]:
# 결측치 시각화
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('결측치 히트맵')
plt.tight_layout()
plt.show()

## 4. Target 변수 분포

In [ ]:
# Target 분포 확인
print(df['Churn'].value_counts())
print(f'\nChurn 비율:\n{df["Churn"].value_counts(normalize=True) * 100}')

In [ ]:
# Target 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 막대 그래프
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Churn 분포 (막대그래프)')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('빈도')

# 파이 차트
df['Churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Churn 분포 (파이차트)')

plt.tight_layout()
plt.show()

## 5. 수치형 변수 분포

In [ ]:
# 수치형 컬럼 선택
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f'수치형 컬럼: {numeric_columns}')

In [ ]:
# 수치형 변수 히스토그램
df[numeric_columns].hist(bins=30, figsize=(15, 10))
plt.suptitle('수치형 변수 분포', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 박스플롯 (이상치 확인)
fig, axes = plt.subplots(1, len(numeric_columns), figsize=(15, 5))
for i, col in enumerate(numeric_columns):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col)
plt.suptitle('수치형 변수 박스플롯', fontsize=16)
plt.tight_layout()
plt.show()

## 6. 범주형 변수 분포

In [ ]:
# 범주형 컬럼 선택
categorical_columns = df.select_dtypes(include='object').columns.tolist()
print(f'범주형 컬럼: {categorical_columns}')

In [ ]:
# 범주형 변수 분포 시각화
fig, axes = plt.subplots(1, len(categorical_columns), figsize=(15, 5))
for i, col in enumerate(categorical_columns):
    df[col].value_counts().plot(kind='bar', ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('범주형 변수 분포', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Target과 변수 관계 분석

In [ ]:
# 수치형 변수 vs Target
fig, axes = plt.subplots(1, len(numeric_columns), figsize=(15, 5))
for i, col in enumerate(numeric_columns):
    sns.boxplot(x='Churn', y=col, data=df, ax=axes[i])
    axes[i].set_title(f'{col} vs Churn')
plt.suptitle('수치형 변수 vs Target', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 범주형 변수 vs Target
fig, axes = plt.subplots(1, len(categorical_columns), figsize=(15, 5))
for i, col in enumerate(categorical_columns):
    pd.crosstab(df[col], df['Churn'], normalize='index').plot(
        kind='bar', ax=axes[i], stacked=True
    )
    axes[i].set_title(f'{col} vs Churn')
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('범주형 변수 vs Target', fontsize=16)
plt.tight_layout()
plt.show()

## 8. 상관관계 분석

In [ ]:
# 상관관계 히트맵
plt.figure(figsize=(10, 8))
corr = df[numeric_columns].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('수치형 변수 상관관계')
plt.tight_layout()
plt.show()

## 9. 이상치 확인

In [ ]:
# 음수값 확인
for col in numeric_columns:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f'{col}: 음수값 {neg_count}개')

In [ ]:
# IQR 기반 이상치 확인
for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'{col}: 이상치 {outliers}개 ({outliers/len(df)*100:.2f}%)')

## 10. EDA 요약

In [ ]:
print('=' * 50)
print('EDA 요약')
print('=' * 50)
print(f'데이터 크기       : {df.shape}')
print(f'수치형 컬럼       : {numeric_columns}')
print(f'범주형 컬럼       : {categorical_columns}')
print(f'결측치 존재 여부  : {df.isnull().any().any()}')
print(f'Target 분포       :\n{df["Churn"].value_counts()}')
print('=' * 50)